# sg-sem: build the data and train the model

The Skill Graph semantic checker (see `SKILL_GRAPH_SEMANTIC_MODEL_SPEC.md` in the repo). This notebook runs on a **Colab GPU** through the VS Code Colab extension:

1. clones the repo and runs our own graph library (Node) to build seed graphs and text jobs,
2. runs an open model **on this GPU** (no API) to write outlines, checker answers, plain-word phrases and gates,
3. assembles the labelled examples, split by graph,
4. fine-tunes the model and reports dev and test metrics against the 0.9 precision bar.

**Run it with `QUICK_LOOK = True` first** (about 10 minutes): it writes a few answers per round so their quality can be checked. Then set it to `False` and run all again; finished answers are kept, so nothing is redone.

Save the notebook when it finishes: its outputs are what gets reviewed.

In [25]:
REPO = "https://github.com/Saad-web-spec/DevScratchPad.git"
BRANCH = "sg-sem"  # the branch with the sg-sem code
GENERATOR = "Qwen/Qwen2.5-7B-Instruct"   # writes the text; runs here on the GPU
BASE = "nreimers/MiniLM-L12-H384-uncased"  # the encoder sg-sem is fine-tuned from
QUICK_LOOK = True                          # True: a few answers per round, to check quality first
TRAIN = not QUICK_LOOK                     # training waits for the full data
RUN = "minilm-1"

## 1. Storage and setup

In [26]:
import os, shutil, subprocess
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/sg-sem"
except Exception as error:  # the VS Code extension may not support Drive mounting
    print("Google Drive not mounted:", error)
    print("Data stays on this runtime only: if it disconnects, the run starts again.")
    ROOT = "/content/sg-sem"
DATA = f"{ROOT}/data-quick" if QUICK_LOOK else f"{ROOT}/data"
RUNS = f"{ROOT}/runs"
os.makedirs(DATA, exist_ok=True); os.makedirs(RUNS, exist_ok=True)
print("data:", DATA)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
data: /content/drive/MyDrive/sg-sem/data-quick


In [27]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
CODE = "/content/DevScratchPad"
if not os.path.exists(CODE):
    !git clone --depth 1 -b {BRANCH} {REPO} {CODE}
else:
    !git -C {CODE} pull --ff-only
!git -C {CODE} log --oneline -1
node = subprocess.run(["node", "--version"], capture_output=True, text=True).stdout.strip() if shutil.which("node") else ""
if not node.startswith(("v20", "v22", "v24")):
    !curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null 2>&1 && apt-get install -y nodejs > /dev/null 2>&1
!node --version
%cd {CODE}
!npm ci --ignore-scripts --no-audit --no-fund --loglevel=error
!pip install -q -r ml/sg-sem/requirements.txt
PIPE = "npx vite-node scripts/semantic/pipeline.ts"
LIMIT = "--limit 24" if QUICK_LOOK else ""

name, memory.total [MiB]
NVIDIA L4, 23034 MiB
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 18 (delta 12), reused 13 (delta 9), pack-reused 0 (from 0)
Unpacking objects: 100% (18/18), 8.57 KiB | 731.00 KiB/s, done.
From https://github.com/Saad-web-spec/DevScratchPad
   37b86ce..650fdb8  sg-sem     -> origin/sg-sem
Updating 37b86ce..650fdb8
Fast-forward
 MEMORY.md                      |   8 +
 ml/sg-sem/make_notebook.py     |  17 +-
 ml/sg-sem/sg_sem.ipynb         | 916 ++++++++++-------------------------------
 ml/sg-sem/sgsem/generate.py    |  21 +-
 scripts/semantic/llmPrompts.ts |  69 +++-
 scripts/semantic/pipeline.ts   |  46 ++-
 6 files changed, 331 insertions(+), 746 deletions(-)
650fdb8 (HEAD -> sg-sem, origin/sg-sem) fix(sg-sem): filter label noise in checker answers and gates; spread the quick look
v20.19.0
/content/DevScratchPad
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇

## 2. Round 1: workflow outlines → seed graphs

In [28]:
!{PIPE} round1 --out {DATA}
!cd ml/sg-sem && python -m sgsem.generate --requests {DATA}/round1.requests.jsonl --outputs {DATA}/round1.outputs.jsonl --model {GENERATOR} {LIMIT}

⠙round1: 200 outline requests → /content/drive/MyDrive/sg-sem/data-quick/round1.requests.jsonl
⠙48 already answered, 24 to go
`torch_dtype` is deprecated! Use `dtype` instead!
I0000 00:00:1792678701.800310   12654 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792678701.871240   12654 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792678704.408435   12654 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set

## 3. Round 2: checker answers, plain-word inputs, gates

In [29]:
!{PIPE} round2 --out {DATA}
!cd ml/sg-sem && python -m sgsem.generate --requests {DATA}/round2.requests.jsonl --outputs {DATA}/round2.outputs.jsonl --model {GENERATOR} {LIMIT}

⠙round2: 69 seeds (55 drafted; rejected {"json":0,"draft":17,"errors":0,"duplicate":0}), 2226 requests {"h1":1992,"h4":166,"h2":68}
why outlines were rejected:
  6 × steps[N].type must be one of orchestrator, worker, verifier, synthesizer, agent, code, approval.
  4 × steps[N].foreach must be <step> or <step>.<key>, e.g. plan.files: the step whose JSON list this step runs over.
  2 × Step … fans out over …, so … must be in its after.
  1 × Step … fans out over …, but … has no … among its keys (brand, model, specifications, price).
  1 × steps[N].id must start with a letter and use only letters, digits, _ or - (got …).
  1 × Step … fans out over …, but … has no … among its keys (make, model, year, price, condition).
  1 × Step … reads from …, which is not a step. Use the ids of other steps, or leave after empty to read the request.
  1 × Step … fans out over …, but … has no … among its keys (description, prerequisites, price).
⠙0 already answered, 24 to go
`torch_dtype` is deprecated! U

## 4. Assemble the labelled examples

In [30]:
!{PIPE} assemble --out {DATA}
print(open(f"{DATA}/stats.json").read())

⠙assemble: 695 examples from 69 seeds; dropped {"h1-no_evidence_pass-length":4,"h1-honest_pass-length":1,"h1-honest_pass-json":1,"h1-resolved_problem_pass-json":1,"h1-resolved_problem_pass-no-fix":1,"h1-buried_problem_pass-not-a-problem":2}
{"h1/train":{"label0":3,"label1":1},"h1/test":{"label0":1,"label1":1},"h4/test":{"label0":5,"label1":2,"label2":3},"h4/train":{"label0":20,"label1":6,"label2":9},"h2/test":{"label1":1,"label0":31},"h2/train":{"label1":3,"label0":239},"h3/test":{"label0":19,"label1":16},"h3/train":{"label0":150,"label1":121},"h2/dev":{"label0":28},"h4/dev":{"label0":1},"h3/dev":{"label0":20,"label1":15}}
⠙{
  "input_version": 1,
  "seeds": 69,
  "examples": 695,
  "by_head_split": {
    "h1/train": {
      "label0": 3,
      "label1": 1
    },
    "h1/test": {
      "label0": 1,
      "label1": 1
    },
    "h4/test": {
      "label0": 5,
      "label1": 2,
      "label2": 3
    },
    "h4/train": {
      "label0": 20,
      "label1": 6,
      "label2": 9
    },
    

In [32]:
# For review: raw answers of the text model, two per kind and style (newest first), and the assembled samples.
import json, collections
requests = {json.loads(l)["id"]: json.loads(l) for l in open(f"{DATA}/round2.requests.jsonl")}
shown = collections.Counter()
for line in reversed(open(f"{DATA}/round2.outputs.jsonl").readlines()):
    row = json.loads(line); request = requests.get(row["id"])
    if request is None:  # an answer to an older prompt
        continue
    kind = (request["kind"], request["meta"].get("style", ""))
    if shown[kind] < 2:
        shown[kind] += 1
        print(f"--- {' '.join(filter(None, kind))} {request['meta']}")
        print(row["text"][:1200])
        print()
print(open(f"{DATA}/samples.md").read()[:12000])


--- h2 {'ref': 'research_product', 'seed': 'compare-products', 'node': 'fact_check', 'pred': 'research_product'}
{"phrase": "the list of products with their specifications and prices"}

--- h2 {'ref': 'score_criterion', 'seed': 'essay-rubric-review', 'node': 'score_check', 'pred': 'score_criterion'}
{"phrase": "the list of scores with their supporting quotes and corrections"}

--- h4 {'seed': 'essay-rubric-review', 'node': 'score_check'}
{"checkable": ["Each score matches at least one supporting quote.", "Every line edit improves readability without altering meaning.", "All scores are integers between 1 and 5."], "partly": ["Every argument score reflects the presence and strength of evidence and reasoning.", "Each structure score should correlate with the clarity and organization of the essay."], "not": ["The essay's overall coherence will enhance its persuasiveness.", "The writer’s engagement level shines through the tone and style choices.", "Future revisions will improve upon curren

## 5. Train (runs when `QUICK_LOOK = False`)

In [33]:
if TRAIN:
    !cd ml/sg-sem && python -m sgsem.train --data {DATA}/examples.jsonl --out {RUNS}/{RUN} --base {BASE}
else:
    print("QUICK_LOOK is on: training waits for the full data.")

QUICK_LOOK is on: training waits for the full data.


In [34]:
if TRAIN:
    report = json.load(open(f"{RUNS}/{RUN}/metrics.json"))
    print(json.dumps({k: report[k] for k in ["test", "rules_baseline_test", "meets_bar_on_test", "minutes"]}, indent=1))
    print(open(f"{RUNS}/{RUN}/errors.md").read()[:6000])